# Shape Measurement with GalSim

Hands-on exploration of galaxy shape measurement using simulated images.
We simulate galaxies with known shear, convolve with a PSF, add noise,
then attempt to recover the shear.

**Key references:**
- Hirata & Seljak (2003) — REGAUSS
- Mandelbaum et al. (2015) — GREAT3 shear testing results
- Sheldon & Huff (2017) — Metacalibration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import galsim

## 1. Simulating a Galaxy Image

Create a simple Sérsic galaxy, apply a known shear, convolve with a
Gaussian PSF, and render to an image.

In [ ]:
# Galaxy parameters
gal_flux = 1e5       # total flux (photons)
gal_hlr = 0.7        # half-light radius (arcsec)
gal_e1_intrinsic = 0.1
gal_e2_intrinsic = 0.05

# Shear to apply (the signal we want to recover)
g1_true = 0.02
g2_true = 0.01

# PSF parameters
psf_fwhm = 0.6       # arcsec (typical ground-based seeing)

# Image parameters
pixel_scale = 0.168  # arcsec/pixel (HSC pixel scale)
stamp_size = 64      # pixels
noise_sigma = 30.0   # sky noise per pixel

In [ ]:
# Create galaxy
gal = galsim.Sersic(n=1.5, half_light_radius=gal_hlr, flux=gal_flux)
gal = gal.shear(e1=gal_e1_intrinsic, e2=gal_e2_intrinsic)
gal = gal.shear(g1=g1_true, g2=g2_true)  # apply lensing shear

# Create PSF
psf = galsim.Gaussian(fwhm=psf_fwhm)

# Convolve
final = galsim.Convolve([gal, psf])

# Draw image
image = final.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)

# Add noise
rng = galsim.BaseDeviate(42)
noise = galsim.GaussianNoise(rng, sigma=noise_sigma)
image.addNoise(noise)

# Also draw the PSF image (needed for shape measurement)
psf_image = psf.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image.array, cmap='viridis', origin='lower')
axes[0].set_title('Galaxy (sheared + PSF + noise)')
axes[1].imshow(psf_image.array, cmap='viridis', origin='lower')
axes[1].set_title('PSF')
plt.tight_layout()
plt.show()

## 2. Measuring Shapes with HSM

GalSim provides the same HSM (Hirata-Seljak-Mandelbaum) algorithms
that are used in the LSST pipelines via `lsst.meas.extensions.shapeHSM`.

In [ ]:
# Measure shape using different estimators
methods = ['REGAUSS', 'KSB', 'BJ', 'LINEAR']

print(f"True applied shear: g1={g1_true}, g2={g2_true}")
print(f"{'Method':<10} {'e1':>10} {'e2':>10}")
print('-' * 32)

for method in methods:
    try:
        result = galsim.hsm.EstimateShear(
            image, psf_image, shear_est=method
        )
        print(f"{method:<10} {result.corrected_e1:>10.4f} {result.corrected_e2:>10.4f}")
    except Exception as e:
        print(f"{method:<10} FAILED: {e}")

## 3. Shear Recovery Test

Simulate many galaxies with random intrinsic shapes but the same applied
shear. The mean measured ellipticity should converge to the true shear
as shape noise averages out.

In [ ]:
n_galaxies = 500
rng = np.random.default_rng(12345)

e1_measured = []
e2_measured = []

for i in range(n_galaxies):
    # Random intrinsic ellipticity
    e_int = rng.rayleigh(0.25)
    while e_int >= 0.9:  # keep physical
        e_int = rng.rayleigh(0.25)
    phi = rng.uniform(0, np.pi)
    e1_int = e_int * np.cos(2 * phi)
    e2_int = e_int * np.sin(2 * phi)

    # Random size
    hlr = rng.uniform(0.4, 1.2)
    flux = 10 ** rng.uniform(4, 5.5)

    gal = galsim.Sersic(n=rng.uniform(0.5, 4.0), half_light_radius=hlr, flux=flux)
    gal = gal.shear(e1=e1_int, e2=e2_int)
    gal = gal.shear(g1=g1_true, g2=g2_true)

    final = galsim.Convolve([gal, psf])
    img = final.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)

    gs_rng = galsim.BaseDeviate(i + 100)
    img.addNoise(galsim.GaussianNoise(gs_rng, sigma=noise_sigma))

    try:
        result = galsim.hsm.EstimateShear(img, psf_image, shear_est='REGAUSS')
        e1_measured.append(result.corrected_e1)
        e2_measured.append(result.corrected_e2)
    except galsim.errors.GalSimHSMError:
        pass  # skip failures

e1_measured = np.array(e1_measured)
e2_measured = np.array(e2_measured)

print(f"Number of successful measurements: {len(e1_measured)}/{n_galaxies}")
print(f"Mean e1 = {e1_measured.mean():.4f} ± {e1_measured.std()/np.sqrt(len(e1_measured)):.4f}  (true g1 = {g1_true})")
print(f"Mean e2 = {e2_measured.mean():.4f} ± {e2_measured.std()/np.sqrt(len(e2_measured)):.4f}  (true g2 = {g2_true})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(e1_measured, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(g1_true, color='red', ls='--', lw=2, label=f'true $g_1$={g1_true}')
axes[0].axvline(e1_measured.mean(), color='orange', ls='-', lw=2, label=f'mean={e1_measured.mean():.4f}')
axes[0].set_xlabel('$e_1$'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].set_title('$e_1$ distribution (REGAUSS)')

axes[1].hist(e2_measured, bins=40, alpha=0.7, color='forestgreen', edgecolor='white')
axes[1].axvline(g2_true, color='red', ls='--', lw=2, label=f'true $g_2$={g2_true}')
axes[1].axvline(e2_measured.mean(), color='orange', ls='-', lw=2, label=f'mean={e2_measured.mean():.4f}')
axes[1].set_xlabel('$e_2$'); axes[1].set_ylabel('Count')
axes[1].legend(); axes[1].set_title('$e_2$ distribution (REGAUSS)')

plt.tight_layout()
plt.savefig('../../figures/shear_recovery_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Multiplicative and Additive Bias

Shear measurement bias is parameterized as:

$$\hat{g}_i = (1 + m_i) g_i^{\text{true}} + c_i$$

where $m$ is the **multiplicative bias** and $c$ is the **additive bias**.

For LSST, the requirements are roughly $|m| < 0.003$ and $|c| < 0.0003$.

**Exercise:** Vary the input shear `g1_true` over a range and fit a linear
relation to estimate $m$ and $c$ for the REGAUSS estimator.

## Next Steps

- Explore how PSF size and ellipticity affect shear recovery
- Compare the four HSM methods on the same simulations
- Move to the LSST pipeline measurement framework